# 07. Human-in-the-Loop, ToolRuntime, and MCP

Learn how LangChain v1 handles **human approval workflows**, **runtime context inside tools**, **context engineering**, and **MCP (Model Context Protocol)**.


## Learning Objectives

This notebook covers:
- **Human-in-the-Loop (HITL):** how to pause agent execution and request approval before a tool call
- **ToolRuntime:** how tools can access runtime context such as user information and session data
- **Context engineering:** techniques for dynamically controlling prompts and tools
- **MCP (Model Context Protocol):** a standardized way to connect tool servers


## 7.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 7.2 Human-in-the-Loop Concepts

Ask for human approval before the agent calls a tool.

### Why is this needed?

Autonomous agents are powerful, but **irreversible actions** such as sending email, deleting files, or processing payments still require human confirmation.

### Workflow

```
Agent → proposes a tool call → [interrupt] → human approves/rejects → tool runs → result is returned
```

In LangChain v1, this is implemented by combining `HumanInTheLoopMiddleware` with `InMemorySaver` (a checkpointer). The checkpointer stores the agent state so the workflow can resume after interruption.


## 7.3 `HumanInTheLoopMiddleware`

`HumanInTheLoopMiddleware` automatically pauses execution on tool calls and waits for human approval. Use it with an `InMemorySaver` checkpointer so interrupted state can be preserved.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """지정된 수신자에게 이메일을 보냅니다."""
    return f"{to}에게 이메일 전송 완료: {subject}"

@tool
def delete_file(path: str) -> str:
    """지정된 경로의 파일을 삭제합니다."""
    return f"파일 삭제 완료: {path}"

# 위험한 도구에만 승인 요구
hitl = HumanInTheLoopMiddleware(interrupt_on={
    "send_email": True,
    "delete_file": True,
})

agent = create_agent(
    model=model,
    tools=[send_email, delete_file],
    system_prompt="당신은 이메일을 보내고 파일을 관리할 수 있는 어시스턴트입니다.",
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("HITL 에이전트 생성 완료")
print("  -> 도구 호출 시 사람의 승인을 위해 중단됩니다")

## 7.4 The `interrupt` and `Command(resume=...)` Pattern

A HITL agent works in two phases:

1. **Phase 1 (`invoke`)**: the agent proposes a tool call and is automatically **interrupted**
2. **Phase 2 (`Command(resume=True)`)**: after a human approves, execution **resumes** with `Command(resume=True)`

To reject a request, use `Command(resume=False)` or provide a different decision.


In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "hitl-demo"}}

# 1단계: 에이전트 실행 -> 도구 호출에서 중단
result = agent.invoke(
    {"messages": [{"role": "user", "content": "bob@example.com에게 제목 '인사', 본문 '안녕하세요 Bob!' 이메일을 보내주세요"}]},
    config={**config, **lf_config},
)

# 중단 상태 확인
print("현재 상태:", result)
print("\n도구 실행 전에 에이전트가 중단되었습니다.")
print("승인하려면 Command(resume=True)를 사용하세요.")

# 2단계: 승인하여 계속 진행
try:
    result = agent.invoke(Command(resume=True), config={**config, **lf_config})
    print("\n승인 후 결과:", result["messages"][-1].content)
except Exception as e:
    print(f"\n참고: HITL 데모는 인터랙티브 환경에서 가장 잘 동작합니다. ({e})")

## 7.5 `ToolRuntime` — Access Runtime Information from a Tool

`ToolRuntime` lets a tool access runtime context such as the current user or session data while it executes.

### Core idea
- Add a `runtime: ToolRuntime[T]` parameter to the tool function
- `T` is a context dataclass defined by the developer
- When you create the agent, set `context_schema=T`, and when invoking the agent, pass `context=T(...)`


In [ ]:
from langchain.tools import ToolRuntime
from dataclasses import dataclass

@dataclass
class UserContext:
    """사용자 정보가 포함된 런타임 컨텍스트."""
    user_id: str
    role: str

@tool
def get_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 프로필 정보를 가져옵니다."""
    ctx = runtime.context
    return f"사용자 ID: {ctx.user_id}, 역할: {ctx.role}"

@tool
def check_permissions(action: str, runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자가 작업에 대한 권한이 있는지 확인합니다."""
    ctx = runtime.context
    if ctx.role == "admin":
        return f"사용자 {ctx.user_id}은(는) '{action}' 권한이 있습니다"
    return f"사용자 {ctx.user_id}은(는) '{action}' 권한이 없습니다"

agent_ctx = create_agent(
    model=model,
    tools=[get_user_profile, check_permissions],
    system_prompt="사용자 프로필과 권한을 확인할 수 있습니다.",
    context_schema=UserContext,
)

result = agent_ctx.invoke(
    {"messages": [{"role": "user", "content": "제가 누구이고 파일을 삭제할 수 있나요?"}]},
    context=UserContext(user_id="user-42", role="admin"),
    config=lf_config,
)
print("결과:", result["messages"][-1].content)

## 7.6 Context Engineering — Dynamic Control of Prompts and Tools

Context engineering is the practice of dynamically shaping the **prompt**, **available tools**, and **message history** given to the agent.

### Common use cases
- Provide a different system prompt depending on user role
- Filter the available tools depending on the situation
- Summarize and reorganize long conversation histories

The `dynamic_prompt` middleware makes it possible to customize the prompt for every request.


In [ ]:
from langchain.agents.middleware import dynamic_prompt

@tool
def basic_search(query: str) -> str:
    """기본 웹 검색을 수행합니다."""
    return f"'{query}'에 대한 기본 검색 결과"

@tool
def advanced_analytics(query: str) -> str:
    """고급 데이터 분석을 수행합니다."""
    return f"'{query}'에 대한 분석 보고서"

# 사용자 역할에 따라 다른 프롬프트와 도구 제공
@dynamic_prompt
def role_based_prompt(request):
    """컨텍스트에 기반하여 프롬프트를 커스터마이즈합니다."""
    return "당신은 전문 어시스턴트입니다. 사용자의 질문에 효율적으로 답변하세요."

agent_ctx_eng = create_agent(
    model=model,
    tools=[basic_search, advanced_analytics],
    middleware=[role_based_prompt],
)

result = agent_ctx_eng.invoke(
    {"messages": [{"role": "user", "content": "머신러닝 트렌드를 검색해 주세요"}]},
    config=lf_config,
)
print("컨텍스트 엔지니어링 결과:", result["messages"][-1].content[:200])

## 7.7 MCP (Model Context Protocol) Integration Overview

**MCP** is a standardized way to connect tool servers.

### Core MCP concepts
- **MCP server**: provides tools through HTTP/SSE or stdio
- **MCP client**: connects to the server and discovers or calls tools
- **Standardization**: any tool can be connected as long as it follows the MCP protocol

### MCP support in LangChain v1
- You can connect to a local MCP server with `mcp.client.stdio.stdio_client()` and `ClientSession`
- `load_mcp_tools(session)` from `langchain-mcp-adapters` converts MCP session tools into LangChain tools


In [ ]:
from pathlib import Path
import tempfile
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

server_path = Path(tempfile.gettempdir()) / "lc_mcp_echo_server.py"
server_path.write_text("""from mcp.server.fastmcp import FastMCP
mcp = FastMCP("echo")
@mcp.tool()
def echo(text: str) -> str:
    return f"Echo: {text}"
if __name__ == "__main__":
    mcp.run(transport="stdio")
""")

async def run_mcp_agent():
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
    async with stdio_client(params) as (read, write), ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        agent = create_agent(model=model, tools=tools, system_prompt="You can use MCP tools.")
        return await agent.ainvoke(
            {"messages": [{"role": "user", "content": "Use the echo tool to repeat hello."}]},
            config=lf_config,
        )

result = await run_mcp_agent()
print(result["messages"][-1].content)


## 7.8 Summary

| Concept | Description | Core API |
|------|------|----------|
| **HITL** | Requests human approval before tool execution | `HumanInTheLoopMiddleware`, `Command(resume=...)` |
| **ToolRuntime** | Gives tools access to runtime context | `ToolRuntime[T]`, `context_schema` |
| **Context engineering** | Dynamically controls prompts and tools | `dynamic_prompt` middleware |
| **MCP** | Standardized tool protocol | `ClientSession + load_mcp_tools()` |

### Next Steps
- The next notebook introduces **multi-agent patterns**.
- You will explore Subagents, Handoffs, Skills, Routers, and other collaboration patterns.
